In [1]:
import re
import json
import time
import sys

# Utils

In [2]:
def custom_progressbar(it, prefix="", size=60, out=sys.stdout): # Python3.6+
    """
    https://stackoverflow.com/questions/3160699/python-progress-bar
    """
    count = len(it)
    start = time.time() # time estimate start
    def show(j):
        x = int(size*j/count)
        # time estimate calculation and string
        elapsed = time.time() - start
        mins, sec = divmod(elapsed, 60) # limited to minutes
        time_str0 = f"{int(mins):02}:{sec:04.1f}"
        remaining = elapsed * (count - j) / j
        mins, sec = divmod(remaining, 60) # limited to minutes
        time_str = f"{int(mins):02}:{sec:04.1f}"
        print(f"{prefix}[{u'█'*x}{('.'*(size-x))}] {j}/{count}, elapsed time {time_str0}, estimated wait time {time_str}", end='\r', file=out, flush=True)
    show(0.1) # avoid div/0
    for i, item in enumerate(it):
        yield item
        show(i+1)
    print("\n", flush=True, file=out)

In [3]:
def longest_common_subsequence(text1, text2): # https://leetcode.com/problems/longest-common-subsequence/
    if True:
        """
        :type text1: str
        :type text2: str
        :rtype: int
        """
        ### Update 03/02/2025: added matched sequence detection part
        ### Update 07/26/2022: all solutions are reviewed
        ### ref) 1143. Longest Common Subsequence: https://leetcode.com/problems/longest-common-subsequence/
        
        ### dynamic programming (bottom-up) with space optimizatoin ##################
        if len(text1) < len(text2): # to make sure text2 is always shorter than text1
            text1, text2 = text2, text1

        prev = [0]*(len(text2)+1) # space complexity: O(min(len(text1), len(text2)))
        curr = [0]*(len(text2)+1) # space complexity: O(min(len(text1), len(text2)))
        for i in range(len(text1)-1, -1, -1): # time complexity: O(len(text1) * len(text2))
            for j in range(len(text2)-1, -1, -1):
                if text1[i] == text2[j]:
                    curr[j] = 1 + prev[j+1]
                else:
                    curr[j] = max(prev[j], curr[j+1])
            prev, curr = curr, prev

        return prev[0]


# Word-Source Dictionary (Trie)
- update: 2026-06-16
- update: 2023-10-03

In [4]:
class Word_Source_Dictionary():
    def __init__(self, word_tokenize=lambda s: re.findall('[0-9a-zA-Z]+', s), is_case_sensitive=False, search_trailing_substring=True):
        self.word_tokenize = word_tokenize
        self.is_case_sensitive = is_case_sensitive
        self.search_trailing_substring = search_trailing_substring
        
        self.trie = {} # initialize trie data structure
        
        self.EOW = '##' # the end of word
        assert len(self.EOW) > 1, 'MUST be more than one character...' # see REF #1
        self.SRC = '$$' # a set of sources
        assert len(self.SRC) > 1, 'MUST be more than one character...' # see REF #1
        self._SRC = '_$' # a set of sources, for search_trailing_substring
        assert len(self._SRC) > 1, 'MUST be more than one character...' # see REF #1

        self.max_num_tokens = 0
        self.counts = {}

    def _add_word(self, word, source, TEG_SRC):
        node = self.trie
        for c in word:
            if c not in node:
                node[c] = {} # create if none, to proceed
            node = node[c]

        node[self.EOW] = word # the end of word. overwrite even if there exists any
        node.setdefault(TEG_SRC, set()).add(source) # a set of sources
        return
        
    def add(self, text, source):
        assert isinstance(text, str) and len(text) > 0, 'Brad error: only non-empty string is allowed....'

        words = self.word_tokenize(text if self.is_case_sensitive else text.lower())
        self.max_num_tokens = max(self.max_num_tokens, len(words))
        for word in words:
            for i in range(len(word) if self.search_trailing_substring else 1): # ex) apple, _pple, __ple, ___le, ____e
                _word = word[i:]
                
                if i == 0:
                    self._add_word(_word, source, self.SRC)
                    if _word not in self.counts:
                        self.counts[_word] = 0
                    self.counts[_word] += 1                    
                else:
                    self._add_word(_word, source, self._SRC)
        return

    def _find_all_leaves(self, node):
        leaves = []
        for k in node.keys():
            if len(k) == 1: # only if a single character, REF #1
                leaves.append(node[k])
                leaves.extend(self._find_all_leaves(node[k])) # recursive search
        return leaves

    def _find_all_common_roots(self, word):
        word = word if self.is_case_sensitive else word.lower()
        
        roots = []
        node = self.trie
        for c in word:
            if c not in node:
                return roots
            node = node[c]
            roots.append(node)

        return roots

    def _find_sources_for_word(self, word, allow_suffix=True, allow_prefix=True, allow_inner_suffix=False, allow_inner_prefix=False):
        word = word if self.is_case_sensitive else word.lower()
        
        sources = set()

        #--- target node and leaf nodes -------------------
        roots = self._find_all_common_roots(word)
        
        if len(roots) == len(word): # ensures the target match
            leaves = self._find_all_leaves(roots[-1]) if allow_suffix else [] # allow matching like xxx_, xxx__, xxx___, ....

            for node in roots[-1:] + leaves: # target node + leaf nodes (if any)
                if self.SRC in node:
                    sources = sources.union(node[self.SRC]) # only extract matching
                    
                if allow_prefix and (self._SRC in node): # allow matching like _xxx, __xxx, ___xxx, ....
                    sources = sources.union(node[self._SRC])

        #--- find root nodes ---------------------
        if allow_inner_suffix or allow_inner_prefix:
            max_len = len(word)
            
            for i in range(max_len if allow_inner_prefix else 1): # _yyy, __yy, ___y
                _word = word[i:]
                
                roots = self._find_all_common_roots(_word)
                roots_ = roots[:(max_len-1)] # exclude only the target node (i.e. word), if any (cf. if len(roots) < len(word), then no effect)

                if allow_inner_suffix:
                    for node in roots_: 
                        if self.SRC in node:
                            sources = sources.union(node[self.SRC])
                elif (len(roots_) == len(_word)) and (self.SRC in node):
                    sources = sources.union(node[self.SRC])

        return sources

    def find_sources(self, texts, allow_suffix=True, allow_prefix=True, allow_inner_suffix=False, allow_inner_prefix=False):
        assert isinstance(texts, list) and len(texts) > 0, 'Brad error: only a non-empty list of texts is allowed....'

        sources = set()
        for text in texts:
            words = self.word_tokenize(text if self.is_case_sensitive else text.lower())

            if len(words) == 0:
                ss = set()
                break

            ss = self._find_sources_for_word(words[0], allow_suffix=allow_suffix, allow_prefix=allow_prefix, allow_inner_suffix=allow_inner_suffix, allow_inner_prefix=allow_inner_prefix)
            for word in words[1:]:
                ss = ss.intersection(self._find_sources_for_word(word, allow_suffix=allow_suffix, allow_prefix=allow_prefix, allow_inner_suffix=allow_inner_suffix, allow_inner_prefix=allow_inner_prefix))

            sources = sources.union(ss)

        return sources


In [5]:
if __name__ == "__main__":
    wsd = Word_Source_Dictionary()

    for text, source in [('Apple Banana',1), ('Cherry',1), ('Cat Dog',2), ('door',3), ('app',4), ('PPL',5), ('TheApplePie',6)]:
        wsd.add(text, source)

    print(wsd.trie)
    # wsd.trie['p']['p']

    print(wsd.find_sources(['apple banana'])) # 'apple' AND 'banana'
    print(wsd.find_sources(['apple', 'door'])) # 'apple' OR 'door'
    print(wsd.find_sources(['apple door'])) # 'apple' AND 'door'

    print(wsd.find_sources(['pp']))
    print(wsd.find_sources(['app'], allow_prefix=False, allow_suffix=False)) # exact matching
    print(wsd.find_sources(['apple']))
    print(wsd.find_sources(['apple'], allow_prefix=False, allow_suffix=False, allow_inner_suffix=True, allow_inner_prefix=True))
    print(wsd.find_sources(['apple'], allow_prefix=True, allow_suffix=True, allow_inner_suffix=True, allow_inner_prefix=True))


{'a': {'p': {'p': {'l': {'e': {'##': 'apple', '$$': {1}, 'p': {'i': {'e': {'##': 'applepie', '_$': {6}}}}}}, '##': 'app', '$$': {4}}}, 'n': {'a': {'n': {'a': {'##': 'anana', '_$': {1}}}, '##': 'ana', '_$': {1}}}, '##': 'a', '_$': {1}, 't': {'##': 'at', '_$': {2}}}, 'p': {'p': {'l': {'e': {'##': 'pple', '_$': {1}, 'p': {'i': {'e': {'##': 'pplepie', '_$': {6}}}}}, '##': 'ppl', '$$': {5}}, '##': 'pp', '_$': {4}}, 'l': {'e': {'##': 'ple', '_$': {1}, 'p': {'i': {'e': {'##': 'plepie', '_$': {6}}}}}, '##': 'pl', '_$': {5}}, '##': 'p', '_$': {4}, 'i': {'e': {'##': 'pie', '_$': {6}}}}, 'l': {'e': {'##': 'le', '_$': {1}, 'p': {'i': {'e': {'##': 'lepie', '_$': {6}}}}}, '##': 'l', '_$': {5}}, 'e': {'##': 'e', '_$': {1, 6}, 'r': {'r': {'y': {'##': 'erry', '_$': {1}}}}, 'a': {'p': {'p': {'l': {'e': {'p': {'i': {'e': {'##': 'eapplepie', '_$': {6}}}}}}}}}, 'p': {'i': {'e': {'##': 'epie', '_$': {6}}}}}, 'b': {'a': {'n': {'a': {'n': {'a': {'##': 'banana', '$$': {1}}}}}}}, 'n': {'a': {'n': {'a': {'##': '

# APPLICATION: Trie for Country-State-City names
- country/state/city names: (https://github.com/dr5hn/countries-states-cities-database)

In [6]:
class CountryStateCity():
    def __init__(
        self, 
        filepath=r'./countries_states_cities_yyyymmdd.json', 
        # additional_data=[{'country_name':None, 'country_code':None, 'state_name':None, 'state_code':None, 'city_name':None},],
        additional_data=[
            {'country_name':'KOREA, REPUBLIC OF', 'country_code':'KR', 'state_name':None, 'state_code':None, 'city_name':None, 'street_name':None},
            {'country_name':'REPUBLIC OF KOREA', 'country_code':'KR', 'state_name':None, 'state_code':None, 'city_name':None, 'street_name':None},
            {'country_name':'Viet nam', 'country_code':'VN', 'state_name':None, 'state_code':None, 'city_name':None, 'street_name':None},
            {'country_name':'Socialist Republic of Viet Nam', 'country_code':'VN', 'state_name':None, 'state_code':None, 'city_name':None, 'street_name':None},
            {'country_name':"Lao People's Democratic Republic", 'country_code':'LA', 'state_name':None, 'state_code':None, 'city_name':None, 'street_name':None},
            {'country_name':'Macau', 'country_code':'MO', 'state_name':None, 'state_code':None, 'city_name':None, 'street_name':None},
            {'country_name':'Macao', 'country_code':'MO', 'state_name':None, 'state_code':None, 'city_name':None, 'street_name':None},
            #--- city names -----------------------------
            {'country_name':'BVI', 'country_code':'VG', 'state_name':'Tortola', 'state_code':None, 'city_name':'Road Town', 'street_name':None},
            {'country_name':'India', 'country_code':'IN', 'state_name':'Maharashtra', 'state_code':'MH', 'city_name':'Bombay', 'street_name':None}, # Bombay, officially renamed Mumbai in 1995, is the capital of Maharashtra and serves as India's financial, commercial, and entertainment powerhouse.
            #--- street names -----------------------------
            {'country_name':'Hongkong', 'country_code':'HK', 'state_name':'Central and Western', 'state_code':None,	'city_name':'Central', 'street_name':"1 QUEEN'S ROAD"}
            ],
        # word_tokenize=lambda s: re.findall('[0-9a-zA-Z]+', unidecode(s)),
        word_tokenize=lambda s: re.findall('[0-9a-zA-Z]+', s),
        ):

        data = json.load(open(filepath, 'r', encoding="utf-8"))

        wsd = Word_Source_Dictionary(word_tokenize=word_tokenize, is_case_sensitive=False, search_trailing_substring=False) # initialize with exad

        map_code_to_country = {}
        _map_ijk_csc = {}
        _map_csc_ijk = {}
        _csc_names = [] # only for 'countries_states_cities_yyyymmdd.json' file
        for i, d in enumerate(custom_progressbar(data)):
            country_name, country_code = d['name'], d['iso2']
            wsd.add(country_name, (i,0)) # country name
            wsd.add(country_code, (i,1)) # country code

            ikey = (country_name, country_code)
            _map_ijk_csc[i] = {'name':ikey}
            _map_csc_ijk[ikey] = {'idx':i}
            map_code_to_country[country_code] = country_name

            for j, s in enumerate(d['states']):
                state_name, state_code = s['name'], s['state_code'] if 'state_code' in s else s['iso2'] # backward compatibility
                wsd.add(state_name, (i,j,0)) # state name
                if (re.search(r'\d', state_code) is None) and (len(state_code) > 1): # add, if there is no digit and len() > 1
                    wsd.add(state_code, (i,j,1)) # state code

                jkey = (state_name, state_code)
                _map_ijk_csc[i][j] = {'name':jkey}
                _map_csc_ijk[ikey][jkey] = {'idx':j}

                for k, c in enumerate(s['cities']):
                    city_name = c['name']
                    wsd.add(city_name, (i,j,k,0)) # city name

                    kkey = (city_name,)
                    _map_ijk_csc[i][j][k] = {'name':kkey}
                    _map_csc_ijk[ikey][jkey][kkey] = {'idx':k}

                    _csc_names.append({'country':country_name, 'country_code':country_code, 'state':state_name, 'state_code':state_name, 'city':city_name})

        for d in custom_progressbar(additional_data):
            ccssc = [d.get('country_name',None), d.get('country_code',None), d.get('state_name',None), d.get('state_code',None), d.get('city_name',None), d.get('street_name',None)]
            # check = [(1 if isinstance(s,str) and len(s.strip()) > 0 else 0) for s in ccssc]
            check = [(1 if isinstance(s,str) and len(s.strip()) > 0 else 0) for s in ccssc[:3] + ccssc[4:]] # exclude 'state_code' from ccssc
            # assert (check == [0,0,0,0,0]) or (check == [1,1,0,0,0]) or (check == [1,1,1,1,0]) or (check == [1,1,1,1,1])
            assert (check == [0,0,0,0,0]) or (check == [1,1,0,0,0]) or (check == [1,1,1,0,0]) or (check == [1,1,1,1,0]) or (check == [1,1,1,1,1])

            if check == [0,0,0,0,0]:
                continue

            country_name, country_code, state_name, state_code, city_name, street_name = ccssc
            ikey = (country_name, country_code)
            jkey = (state_name, state_code)
            kkey = (city_name,)
            lkey = (street_name,)

            if country_code not in map_code_to_country:
                map_code_to_country[country_code] = country_name

            #--- country ------------------------------------
            if ikey in _map_csc_ijk:
                i = _map_csc_ijk[ikey]['idx']
            else:
                i = len(_map_ijk_csc)
                _map_ijk_csc[i] = {'name':ikey}
                _map_csc_ijk[ikey] = {'idx':i}
            wsd.add(country_name, (i,0)) # country name
            wsd.add(country_code, (i,1)) # country code

            if check == [1,1,0,0,0]:
                continue

            #--- state ------------------------------------
            if jkey in _map_csc_ijk[ikey]:
                j = _map_csc_ijk[ikey][jkey]['idx']
            else:
                j = len(_map_ijk_csc[i])
                _map_ijk_csc[i][j] = {'name':jkey}
                _map_csc_ijk[ikey][jkey] = {'idx':j}
            wsd.add(state_name, (i,j,0)) # state name
            # wsd.add(state_code, (i,j)) # state code
            if isinstance(state_code,str) and re.search(r'\d', state_code) is None:
                wsd.add(state_code, (i,j,1)) # state code

            if check == [1,1,1,0,0]:
                continue

            #--- city ------------------------------------
            if kkey in _map_csc_ijk[ikey][jkey]:
                k = _map_csc_ijk[ikey][jkey][kkey]['idx']
            else:
                k = len(_map_ijk_csc[i][j])
                _map_ijk_csc[i][j][k] = {'name':kkey}
                _map_csc_ijk[ikey][jkey][kkey] = {'idx':k}
            wsd.add(city_name, (i,j,k,0)) # city name

            if check == [1,1,1,1,0]:
                continue

            #--- street ------------------------------------
            if lkey in _map_csc_ijk[ikey][jkey][kkey]:
                l = _map_csc_ijk[ikey][jkey][kkey][lkey]['idx']
            else:
                l = len(_map_ijk_csc[i][j][k])
                _map_ijk_csc[i][j][k][l] = {'name':lkey}
                _map_csc_ijk[ikey][jkey][kkey][lkey] = {'idx':l}
            wsd.add(street_name, (i,j,k,l,0)) # street name

            if check == [1,1,1,1,1]:
                continue

        del data

        self.wsd = wsd
        self.word_tokenize = word_tokenize
        self.max_num_tokens = wsd.max_num_tokens
        self.map_code_to_country = map_code_to_country
        self._map_ijk_csc = _map_ijk_csc
        self._csc_names = _csc_names
        return

    def get_search_result(self, search_name):
        results = self.wsd.find_sources([search_name], allow_prefix=False, allow_suffix=False) # exact matching

        search_results = []
        for ijkl in results:
            full_path = []
            node = self._map_ijk_csc
            for n in ijkl[:-1]:
                node = node[n]
                full_path.append(node['name'])

            search_results.append({'full_path':full_path, 'pos':ijkl[-1]})

        # search_results = sorted(search_results, key=lambda x: (len(x['full_path']), x['full_path']))
        # search_results = sorted(search_results, key=lambda x: (len(x['full_path']), len(x['full_path'][-1][0]), x['full_path'])) # (depth, len_name, ...)
        search_results = sorted(search_results, key=lambda x: (len(x['full_path'][-1][0]), len(x['full_path']), x['full_path'])) # (depth, len_name, ...)

        return search_results

    # def save_country_state_city(self, filepath=r'country_state_city_names.xlsx'):
    #     save_dataframes_in_excel(filepath, [('data', pd.DataFrame(self._csc_names))])


In [7]:
if __name__=='__main__':
    # !pip install Unidecode
    from unidecode import unidecode

    filepath = r'../../datasets/fake_transaction_records/countries_states_cities_20260509.json' # https://github.com/dr5hn/countries-states-cities-database/tree/master/json
    csc = CountryStateCity(filepath=filepath, word_tokenize=lambda s: re.findall('[0-9a-zA-Z]+', unidecode(s)))
    # csc.save_country_state_city() # save

    # sorted(list(csc.wsd.counts.items()), key=lambda x: x[-1])

    #----------------------------------------------------------------
    search_name = 'korea'
    search_results = csc.get_search_result(search_name)
    for sr in reversed(search_results):
        print(sr)

[████████████████████████████████████████████████████████████] 250/250, elapsed time 00:02.1, estimated wait time 00:00.0

[████████████████████████████████████████████████████████████] 10/10, elapsed time 00:00.0, estimated wait time 00:00.00

{'full_path': [('KOREA, REPUBLIC OF', 'KR')], 'pos': 0}
{'full_path': [('REPUBLIC OF KOREA', 'KR')], 'pos': 0}
{'full_path': [('South Korea', 'KR')], 'pos': 0}
{'full_path': [('North Korea', 'KP')], 'pos': 0}


# APPLICATION: Identify Country from Address

In [8]:
def get_trie_for_all_cadidates(
    text_address=None, csc=None, reverse_stop_words=set(), skip_conditions=lambda token: len(token) <= 1,
    EXP_SCORE1_OVER_ALL=0.7,
    ):
    # !pip install Unidecode
    from unidecode import unidecode

    tokens_orig = csc.word_tokenize(text_address)

    #--------------------------------------------------------------
    if reverse_stop_words: # truncate text if stop words are seen for the first time in reverse order
        tokens_trunc = []
        for t in reversed(tokens_orig):
            if unidecode(t).lower() in reverse_stop_words:
                break
            tokens_trunc.append(t)
        tokens = tokens_trunc[::-1] # overwrite
    else:
        tokens = tokens_orig

    #--- main -----------------------------------------------------
    trie = {}
    checked = [set() for _ in range(len(tokens))]
    num_all_token_chars = sum(len(t) for t in tokens)

    for ntoken in range(min(csc.max_num_tokens, len(tokens)), 0, -1):
        for ipos in range(len(tokens)-1, -1 + (ntoken - 1), -1):
            ttttt = tokens[(ipos + 1 - ntoken):(ipos + 1)] #  subset of tokens

            _tokens_ = ' '.join(ttttt)

            #--- skip --------------------------------------
            # if skip_conditions(_tokens_):
            if skip_conditions(_tokens_.lower()):
                continue

            #--- find candidate matches ----------------------------
            candidates = csc.get_search_result(_tokens_)

            if len(candidates) == 0:
                continue

            #---------------------------------------------
            text1 = re.sub(r'[^0-9a-zA-Z]', '', unidecode(_tokens_).lower())

            num_matches_1 = sum(len(t) for t in ttttt)
            score1_over_all = num_matches_1 / num_all_token_chars

            seen = {}
            for candidate in candidates:
                country_state_city, pos = candidate['full_path'], candidate['pos']
                entity_name = country_state_city[-1][pos]

                country_state_city_normalized = [cc for cc in country_state_city] # [(name, code), ...]
                country_state_city_normalized[0] = tuple([country_state_city[0][1], csc.map_code_to_country[country_state_city[0][1]]]) # [(code, name), ...]: normalize country name

                country = country_state_city_normalized[0][0] # country code

                #--- skip if already checked -------------------------
                if country in checked[ipos]:
                    continue

                #--- calculate similarity score -------------------
                text2 = re.sub(r'[^0-9a-zA-Z]', '', unidecode(entity_name).lower())
                num_matches_2 = longest_common_subsequence(text1, text2)
                # score1 = num_matches_2 / len(text1)
                score2 = num_matches_2 / len(text2)

                #--- calculate score ------------------------------------------

                if (country not in seen) or (len(country_state_city_normalized) > 1 and len(country_state_city_normalized) == seen[country]):
                    #--- initialize trie --------------------------------
                    node = trie
                    for r in country_state_city_normalized:
                        if r not in node:
                            node[r] = {'cnt':0, 'score':0, 'depth':[], 'num_matches_1':[], 'score2s':[]}
                        node = node[r]

                    score_position = sum(((ipos + 1 - x) / len(tokens))**1 for x in range(ntoken))
                    score_region = 1 - (len(country_state_city_normalized) - 1) / 5 # always should be > 0
                    score_match1_all = score1_over_all ** EXP_SCORE1_OVER_ALL
                    score_match2 = score2

                    node_score = score_position * score_region * score_match1_all * score_match2

                    node['cnt'] += 1 # add count for the last node
                    node['score'] += node_score
                    node['depth'].append(len(country_state_city_normalized))
                    node['num_matches_1'].append(num_matches_1)
                    node['score2s'].append(score2)

                    seen[country] = len(country_state_city_normalized)

            for iii in range(ipos + 1 - ntoken, ipos + 1):
                checked[iii] = checked[iii].union(list(seen.keys())) # check if detected

    #--------------------------------------------------------------------------------
    def add_aggregate_fields(node, level=0):
        out = {'max_sum':0, 'tot_cnt':0, 'depth_all':[], 'tot_num_matches':0, 'score2s_all':[]} # initialize
        for k in node.keys():
            if isinstance(k, tuple):
                out2 = add_aggregate_fields(node[k], level=level + 1)
                if (out['max_sum'], out['tot_cnt']) < (out2['max_sum'], out2['tot_cnt']):
                    out = out2 # update

        ans = {
            'max_sum': node.get('score',0) + out['max_sum'],
            'tot_cnt': node.get('cnt',0) + out['tot_cnt'],
            'depth_all': node.get('depth',[]) + out['depth_all'],
            'tot_num_matches': sum(node.get('num_matches_1',[])) + out['tot_num_matches'],
            'score2s_all': node.get('score2s',[]) + out['score2s_all'],
        }

        if level in [0,1]:
            for k in ans.keys():
                node[k] = ans[k] # add new field to each trie node

        return ans

    _ = add_aggregate_fields(trie, level=0)

    #--------------------------------------------------------------------------------
    def get_log(node):
        head_line = []
        logs = []
        for k in node.keys():
            if isinstance(k, tuple):
                logs2 = get_log(node[k])
                for i, line in enumerate(logs2):
                    if i == 0:
                        logs.append(f'{k}: ' + line)
                    else:
                        logs.append('\t\t' + line)
            else:
                head_line.append(f'{k}:{node[k]}')

        if head_line:
            logs = [', '.join(head_line)] + logs

        return logs

    detail = '\n'.join(get_log(trie))

    #--------------------------------------------------------------------------------
    # country_codes = list(set(k[1] for k in trie.keys() if isinstance(k,tuple) and trie[k]['max_sum'] == trie['max_sum'] and trie[k]['tot_cnt'] == trie['tot_cnt']))
    country_codes = list(set(k[0] for k in trie.keys() if isinstance(k,tuple) and trie[k]['max_sum'] == trie['max_sum'] and trie[k]['tot_cnt'] == trie['tot_cnt']))
    country_names = [csc.map_code_to_country[c] for c in country_codes]

    country_codes = country_codes[0] if len(country_codes) == 1 else country_codes
    country_names = country_names[0] if len(country_names) == 1 else country_names

    return {
        'trie':trie, 'detail':detail,
        'max_sum_score':trie['max_sum'], 'total_count':trie['tot_cnt'], 'depth_all':sorted(set(trie['depth_all'])), 'tot_num_matches':trie['tot_num_matches'], 
        'min_score2':min(trie['score2s_all']) if trie['score2s_all'] else None, 
        'max_score2':max(trie['score2s_all']) if trie['score2s_all'] else None, 
        'country_codes':country_codes, 'country_names':country_names,
        'checked':checked,
        'tokens':tokens,
    }


In [9]:
skip_words = set([w.lower() for w in [
    'street','road','avenue',
    'plaza','building','branch',
    'room','floor','level',
    'co ltd','co limited',
    'development','bank',
    'of','and',#'or',
]])

In [10]:
if __name__=='__main__':
    # !pip install Unidecode
    from unidecode import unidecode

    text_address = r'New York Public Library - Stephen A. Schwarzman Building, 476 5th Ave, New York, NY 10018'

    # out = get_trie_for_all_cadidates(text_address=text_address, csc=csc, reverse_stop_words=skip_words)
    out0 = get_trie_for_all_cadidates(text_address=text_address, csc=csc)
    out = get_trie_for_all_cadidates(text_address=text_address, csc=csc, skip_conditions=lambda token: len(token) <= 1 or (token in skip_words))

    print(out0['country_names'])
    print(out['country_names'])
    # print(out['trie'][('PA','Panama')])
    # print(out['trie'][('US','United States')][('Florida', 'FL')])
    # print(out['checked'])
    # print(out['detail'])



United States
United States
